# Base Model

## Imports

In [2]:
import random
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Loading Data

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [4]:
print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

image, label = train_dataset[0]
print("\nImage shape:", image.shape)
print("Label:", label)

Training samples: 60000
Test samples: 10000

Image shape: torch.Size([1, 28, 28])
Label: 5


## Config Settings

In [5]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)

CUDA available: True
CUDA version: 12.1
Torch version: 2.5.1+cu121


In [6]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


## Model Code

### Base Model Code

In [7]:
class ShallowNN(nn.Module):
    def __init__(self, hidden_size, p=0):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 10)
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [8]:
class Trainer:
    def __init__(self, hidden_size, learning_rate, epochs, batch_size, train_data, test_data, p=0):
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = ShallowNN(hidden_size=hidden_size, p=p).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()

        self.init_data_loaders(train_data, test_data)

    def init_data_loaders(self, train_data, test_data):
        # TODO: add param to pass dataset (normal or poisoned)

        self.train_loader = DataLoader(
            train_data,
            batch_size=self.batch_size,
            shuffle=True
        )

        self.test_loader = DataLoader(
            test_data,
            batch_size=1000,
            shuffle=False
        )

    def train(self, debug=False):
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            for data, target in self.train_loader:
                data, target = data.to(self.device), target.to(self.device)

                self.optimizer.zero_grad()
                output = self.model(data)
                loss = self.criterion(output, target)
                loss.backward()
                self.optimizer.step()

            acc = self.evaluate()

            if debug:
                print(f"Epoch {epoch}/{self.epochs} | Test Accuracy: {acc:.4f}")

    def evaluate(self):
        self.model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in self.test_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                pred = output.argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)
        return correct / total

### Model Addons

#### Mislabelling

In [ ]:
def add_label_noise(dataset, noise_ratio=0.45, num_classes=10, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    n = len(dataset.targets)
    n_noisy = int(noise_ratio * n)

    noisy_indices = torch.randperm(n)[:n_noisy]

    original = dataset.targets[noisy_indices]
    noise = torch.randint(0, num_classes - 1, size=(n_noisy,))
    noise = (noise + (noise >= original)).long()

    dataset.targets[noisy_indices] = noise

    return dataset

#### Mixup Augmentation

In [ ]:
# TODO: Mixup augmentation

#### Cutout Augmentation

In [13]:
class CutoutAugmentation():
    def __init__(self, K=16):
        self.K = K

    def __call__(self, img):
        # tensor shape: c, h, w
        if random.random() < 0.5:
            return img

        C, H, W = img.shape
        half = self.K // 2

        # random center
        cx = random.randint(0, W - 1)
        cy = random.randint(0, H - 1)

        # bounding box
        x1 = max(cx - half, 0)
        x2 = min(cx + half, W)
        y1 = max(cy - half, 0)
        y2 = min(cy + half, H)

        img[:, y1:y2, x1:x2] = 0.0
        return img

#### Standard Augmentation

In [15]:
class RandomShift():
    def __init__(self, K):
        self.K = K

    def __call__(self, img):
        k1 = random.randint(-self.K, self.K)
        k2 = random.randint(-self.K, self.K)

        C, H, W = img.shape
        shifted = torch.zeros_like(img)

        h_start = max(0, k1)
        h_end = min(H, H + k1)
        w_start = max(0, k2)
        w_end = min(W, W + k2)

        orig_h_start = max(0, -k1)
        orig_h_end = orig_h_start + (h_end - h_start)
        orig_w_start = max(0, -k2)
        orig_w_end = orig_w_start + (w_end - w_start)

        shifted[:, h_start:h_end, w_start:w_end] = img[:, orig_h_start:orig_h_end, orig_w_start:orig_w_end]

        img = shifted

        # horizontal flip
        # if random.random() < 0.5:
        #     img = torch.flip(img, dims=[2])

        return img

## Testing

### Base Model & Dropout

In [10]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.9189
Epoch 2/10 | Test Accuracy: 0.9371
Epoch 3/10 | Test Accuracy: 0.9487
Epoch 4/10 | Test Accuracy: 0.9543
Epoch 5/10 | Test Accuracy: 0.9593
Epoch 6/10 | Test Accuracy: 0.9624
Epoch 7/10 | Test Accuracy: 0.9645
Epoch 8/10 | Test Accuracy: 0.9654
Epoch 9/10 | Test Accuracy: 0.9666
Epoch 10/10 | Test Accuracy: 0.9704


In [14]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0.4
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.9188
Epoch 2/10 | Test Accuracy: 0.9348
Epoch 3/10 | Test Accuracy: 0.9458
Epoch 4/10 | Test Accuracy: 0.9509
Epoch 5/10 | Test Accuracy: 0.9552
Epoch 6/10 | Test Accuracy: 0.9584
Epoch 7/10 | Test Accuracy: 0.9627
Epoch 8/10 | Test Accuracy: 0.9627
Epoch 9/10 | Test Accuracy: 0.9654
Epoch 10/10 | Test Accuracy: 0.9646


In [17]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=train_dataset,
    test_data=test_dataset,
    p=0.8
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.8986
Epoch 2/10 | Test Accuracy: 0.9125
Epoch 3/10 | Test Accuracy: 0.9169
Epoch 4/10 | Test Accuracy: 0.9223
Epoch 5/10 | Test Accuracy: 0.9241
Epoch 6/10 | Test Accuracy: 0.9293
Epoch 7/10 | Test Accuracy: 0.9307
Epoch 8/10 | Test Accuracy: 0.9322
Epoch 9/10 | Test Accuracy: 0.9333
Epoch 10/10 | Test Accuracy: 0.9330


In [21]:
print(f"Test Accuracy: {trainer.evaluate()}")

Test Accuracy: 0.9676


### Noisy Labels

In [42]:
label_noise_ratio = 0.45

noisy_train_dataset = add_label_noise(copy.deepcopy(train_dataset), noise_ratio=label_noise_ratio)

In [43]:
# sanity testing to see noisy label thing worked
changed = (noisy_train_dataset.targets != train_dataset.targets).sum().item()
noise_fraction = changed / len(noisy_train_dataset)

print("Expected noise fraction:", label_noise_ratio)
print("Noise fraction:", noise_fraction)

Expected noise fraction: 0.45
Noise fraction: 0.45


In [44]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=noisy_train_dataset,
    test_data=test_dataset,
    p=0
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.9099
Epoch 2/10 | Test Accuracy: 0.9294
Epoch 3/10 | Test Accuracy: 0.9358
Epoch 4/10 | Test Accuracy: 0.9422
Epoch 5/10 | Test Accuracy: 0.9386
Epoch 6/10 | Test Accuracy: 0.9467
Epoch 7/10 | Test Accuracy: 0.9473
Epoch 8/10 | Test Accuracy: 0.9464
Epoch 9/10 | Test Accuracy: 0.9481
Epoch 10/10 | Test Accuracy: 0.9466


### Mixup Aug. Testing

### Cutout Aug. Testing

In [21]:
cutout_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    CutoutAugmentation(K=16)
])

cutout_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=cutout_transform
)

In [22]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=cutout_train_dataset,
    test_data=test_dataset,
    p=0
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.9125
Epoch 2/10 | Test Accuracy: 0.9296
Epoch 3/10 | Test Accuracy: 0.9428
Epoch 4/10 | Test Accuracy: 0.9484
Epoch 5/10 | Test Accuracy: 0.9533
Epoch 6/10 | Test Accuracy: 0.9566
Epoch 7/10 | Test Accuracy: 0.9594
Epoch 8/10 | Test Accuracy: 0.9642
Epoch 9/10 | Test Accuracy: 0.9658
Epoch 10/10 | Test Accuracy: 0.9674


### Standard Aug. Testing

In [19]:
standard_aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    RandomShift(K=4)
])

standard_aug_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=False,
    transform=standard_aug_transform
)

In [20]:
trainer = Trainer(
    hidden_size=64,
    learning_rate=1e-3,
    epochs=10,
    batch_size=512,
    train_data=standard_aug_train_dataset,
    test_data=test_dataset,
    p=0
)

trainer.train(debug=True)

Epoch 1/10 | Test Accuracy: 0.8246
Epoch 2/10 | Test Accuracy: 0.8833
Epoch 3/10 | Test Accuracy: 0.9075
Epoch 4/10 | Test Accuracy: 0.9156
Epoch 5/10 | Test Accuracy: 0.9279
Epoch 6/10 | Test Accuracy: 0.9329
Epoch 7/10 | Test Accuracy: 0.9358
Epoch 8/10 | Test Accuracy: 0.9391
Epoch 9/10 | Test Accuracy: 0.9369
Epoch 10/10 | Test Accuracy: 0.9463
